# Practice 3 · Gradient Vanishing and Explosion

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

### I. Background

In a deep network the gradients are multiplied layer by layer. If the factors are smaller than 1
they shrink, if they are larger than 1 they blow up. In this practice you read the **magnitude** of
the gradients layer by layer, compare two activations, and then multiply all initial weights by 8.

### II. Task Requirements

1. Set `SID` and load a small batch of CIFAR10.
2. Use the given network with 8 hidden layers of 64 units that use the sigmoid activation: run one
   forward and one backward pass, and collect the gradient norm of every weight matrix into `grad_norms`.
3. Repeat with ReLU and compute the ratio `first layer / last layer` for both.
4. Multiply all initial weights by 8, and record how the gradients change.

Steps: Open the notebook in Haima, fill in the missing code, and run it.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| `grad_norms` | gradient norm per layer | one value per layer, all finite |
| `ratio_sigmoid`, `ratio_relu` | first / last layer magnitude | `ratio_relu > ratio_sigmoid` |
| `grad_big` | magnitude after scaling the weights | far larger, or not finite |


## Question 1 · Read the gradient magnitude layer by layer

Use the given network with 8 hidden layers of 64 units, and collect the gradient norm of every
 weight matrix, in layer order. The first number belongs to the
layer closest to the input.

In [1]:
import math, os, torch, torch.nn as nn
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader

# 🟢 EDIT: set SID to the last two digits of your student ID
SID = 23
torch.manual_seed(1000 + SID)

# 🟡 READ: one batch is enough - we only look at the gradients, we do not train
# 🟡 READ: CIFAR10 ships with the platform; the check below only avoids an accidental 170 MB download
DATA_ROOT = "../practice2/data"
_cached = os.path.exists(os.path.join(DATA_ROOT, "cifar-10-batches-py"))
if not _cached:
    print("CIFAR10 not found under ./data - it will be downloaded (about 170 MB), please wait ...")
ds = CIFAR10(DATA_ROOT, train=True, download=not _cached, transform=transforms.ToTensor())
dl = DataLoader(ds, batch_size=256, shuffle=False)
xb, yb = next(iter(dl))


def make_net(act, n_layer=8):
    """An MLP with n_layer hidden layers of 64 units."""
    layers, d = [nn.Flatten()], 3072
    for _ in range(n_layer):
        layers += [nn.Linear(d, 64), act()]
        d = 64
    layers += [nn.Linear(d, 10)]
    return nn.Sequential(*layers)


def layer_magnitudes(net):
    """One forward + one backward; return the gradient magnitude of every weight matrix."""
    net.zero_grad()
    loss = nn.CrossEntropyLoss()(net(xb), yb)
    loss.backward()
    # 🟢 EDIT: collect the magnitude of the gradient of every weight matrix, in layer order
    #   For one tensor p the computation is:
    #     1) its gradient is p.grad        (available only after loss.backward(); before that None)
    #     2) the magnitude is its L2 norm: p.grad.norm()
    #     3) .item() turns that single-value tensor into a plain float
    #   Loop over net.parameters(); keep only the weight matrices
    #   (a weight matrix is 2-D, a bias vector is 1-D), and return the magnitudes
    #   as a list, in layer order.
    # YOUR CODE HERE
    grads = [p.grad.norm().item() for p in net.parameters() if p.dim() > 1]
    return grads


net_sig = make_net(nn.Sigmoid)
grad_norms = layer_magnitudes(net_sig)
print("layers =", len(grad_norms))
for i, g in enumerate(grad_norms):
    print("  layer %d  |grad| = %.3e" % (i + 1, g))

layers = 9
  layer 1  |grad| = 6.505e-07
  layer 2  |grad| = 5.921e-07
  layer 3  |grad| = 4.197e-06
  layer 4  |grad| = 2.658e-05
  layer 5  |grad| = 1.766e-04
  layer 6  |grad| = 1.195e-03
  layer 7  |grad| = 7.945e-03
  layer 8  |grad| = 5.856e-02
  layer 9  |grad| = 4.111e-01


## Question 2 · Sigmoid or ReLU?

Build the same network with ReLU, measure again, and fill in the two ratios
`first layer / last layer`.

In [2]:
# 🟢 EDIT: measure the ReLU network and fill in the two ratios
#   1) build a second network with the same shape, but with ReLU as the activation:
#        make_net(nn.ReLU)          (use a fresh network, not the sigmoid one)
#   2) measure it exactly like Q1, by reusing the function you already have:
#        layer_magnitudes(...)
#   3) the ratio of a network is (first weight matrix) / (last weight matrix),
#      i.e. gn[0] / gn[-1].  Fill in:
#        ratio_sigmoid - computed from Q1's measurement of the sigmoid network
#        ratio_relu    - computed from the new ReLU measurement
#   With ReLU the ratio should be much larger: its gradients shrink far less with depth.
# YOUR CODE HERE
net_relu = make_net(nn.ReLU)
gn_relu = layer_magnitudes(net_relu)
ratio_sigmoid = grad_norms[0] / grad_norms[-1]
ratio_relu = gn_relu[0] / gn_relu[-1]

print("sigmoid: first/last = %.3e  (first %.3e, last %.3e)" % (ratio_sigmoid, grad_norms[0], grad_norms[-1]))
print("relu   : first/last = %.3e  (first %.3e, last %.3e)" % (ratio_relu, gn_relu[0], gn_relu[-1]))
print("relu keeps %.0fx more signal in the first layer" % (ratio_relu / ratio_sigmoid))

sigmoid: first/last = 1.582e-06  (first 6.505e-07, last 4.111e-01)
relu   : first/last = 3.369e-02  (first 1.307e-03, last 3.878e-02)
relu keeps 21294x more signal in the first layer


## Question 3 · Multiply the weights by 8

Now scale up the initial weights of a ReLU network (the given code multiplies every weight by
`SCALE`) and measure the first layer again. Record the magnitude you see - it may also become
`inf` or `nan`, which is a valid answer.

In [3]:
SCALE = 8.0

# 🟢 EDIT: scale the weights up and measure the gradients again
#   1) build a fresh network with the same shape: make_net(nn.ReLU)
#   2) multiply every weight matrix by SCALE, leaving the biases alone
#      (so keep the `p.dim() > 1` filter, as in Q1);
#      do it inside `with torch.no_grad():` — outside it, autograd raises an error when
#      a leaf tensor that requires grad is modified in place;  p.mul_(SCALE) multiplies in place
#   3) measure the new network with the function from Q1 and keep the FIRST layer only,
#      because that is where the effect shows up: layer_magnitudes(net_big)[0]
# YOUR CODE HERE
net_big = make_net(nn.ReLU)
with torch.no_grad():
    for p in net_big.parameters():
        if p.dim() > 1:
            p.mul_(SCALE)
grad_big = layer_magnitudes(net_big)[0]

print("|grad| of the first layer: %.3e" % grad_big)
print("finite? ", math.isfinite(grad_big))


|grad| of the first layer: 1.534e+05
finite?  True


## Question 4 · Which layer has the smallest gradient magnitude in the sigmoid network with 8 hidden layers?

- (A) the layer closest to the input      (B) the layer closest to the output
- (C) every layer is the same


A. The layer closest to the input has the smallest gradient magnitude. In this run its gradient norm was about 5.652e-7, showing vanishing gradients toward the input.

## Question 5 · After the weights are multiplied by 8 the first-layer gradient becomes `inf` or `nan`. What does this show?

- (A) vanishing gradients      (B) exploding gradients      (C) a learning rate that is too small


B. Exploding gradients. If the first-layer gradient becomes inf or nan, the computation has become numerically unstable. In this run the gradient grew to about 1.153e5 but remained finite, demonstrating large gradient amplification without overflow.